In [ ]:
# EXPERIMENT 2 — Unfreeze GPT-2 TOP 4 layers + embeddings

# ── CELL 1: Install ────────────────────────────────────────
# !pip install TTS torch torchaudio trainer

# ── CELL 2: Imports ───────────────────────────────────────
import torch
import matplotlib.pyplot as plt
import json, os

from trainer import Trainer, TrainerArgs
from TTS.tts.configs.xtts_config import XttsConfig
from TTS.tts.datasets import load_tts_samples
from TTS.tts.models.xtts import Xtts

print("=" * 50)
print("EXPERIMENT 2: Unfreeze GPT-2 TOP 4 blocks + Embeddings")
print("=" * 50)

# ── CELL 3: Paths ─────────────────────────────────────────
XTTS_CHECKPOINT = "/root/.local/share/tts/tts_models--multilingual--multi-dataset--xtts_v2/"
DATASET_PATH    = "xtts_dataset/"
OUTPUT_PATH     = "exp2_output/"
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── CELL 4: Config ────────────────────────────────────────
config = XttsConfig()
config.load_json(os.path.join(XTTS_CHECKPOINT, "config.json"))

config.output_path      = OUTPUT_PATH
config.epochs           = 20
config.batch_size       = 4
config.eval_batch_size  = 2
config.lr               = 5e-5        # lower than exp1
config.print_step       = 50
config.save_step        = 500
config.save_checkpoints = True
config.print_eval       = True

print(f"Epochs     : {config.epochs}")
print(f"LR         : {config.lr}")
print(f"Batch size : {config.batch_size}")

# ── CELL 5: Load dataset ──────────────────────────────────
train_samples, eval_samples = load_tts_samples(
    datasets=[{
        "formatter"       : "ljspeech",
        "dataset_name"    : "urdu_tts",
        "path"            : DATASET_PATH,
        "meta_file_train" : "train/metadata.csv",
        "meta_file_val"   : "val/metadata.csv",
        "language"        : "ur",
        "ignored_speakers": None,
    }],
    eval_split=True,
)
print(f"\nTrain samples : {len(train_samples)}")
print(f"Val samples   : {len(eval_samples)}")

# ── CELL 6: Load model ────────────────────────────────────
model = Xtts.init_from_config(config)
model.load_checkpoint(config, checkpoint_dir=XTTS_CHECKPOINT, eval=False)
print("\nPretrained XTTS loaded ✅")

# ── CELL 7: Freeze all, unfreeze top 4 GPT-2 blocks ──────
for param in model.parameters():
    param.requires_grad = False

TOP_N_BLOCKS = 4
unfrozen_layers = []

for name, param in model.named_parameters():
    # Unfreeze text embeddings always
    if "text_embedding" in name:
        param.requires_grad = True
        unfrozen_layers.append(name)

    # Unfreeze top N GPT-2 transformer blocks
    if "gpt" in name:
        for i in range(TOP_N_BLOCKS):
            block_id = 24 - i   # XTTS GPT has 24 blocks, unfreeze last N
            if f"h.{block_id}." in name:
                param.requires_grad = True
                unfrozen_layers.append(name)

# Count params
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"\nFreezing Strategy:")
print(f"  Unfrozen : text embeddings + GPT-2 top {TOP_N_BLOCKS} blocks")
print(f"  Total params     : {total_params:,}")
print(f"  Trainable params : {trainable_params:,}")
print(f"  Frozen params    : {total_params - trainable_params:,}")
print(f"  Trainable %      : {100 * trainable_params / total_params:.2f}%")

# ── CELL 8: Train ─────────────────────────────────────────
print("\n🚀 Starting Experiment 2 training...")

trainer = Trainer(
    TrainerArgs(restore_path=None, skip_train_epoch=False),
    config,
    output_path=OUTPUT_PATH,
    model=model,
    train_samples=train_samples,
    eval_samples=eval_samples,
)

trainer.fit()
print("\n✅ Experiment 2 complete!")

# ── CELL 9: Plot loss curve ───────────────────────────────
log_path = os.path.join(OUTPUT_PATH, "trainer_0_log.json")
if os.path.exists(log_path):
    with open(log_path) as f:
        logs = json.load(f)

    steps  = [l["step"] for l in logs if "loss" in l]
    losses = [l["loss"] for l in logs if "loss" in l]

    plt.figure(figsize=(10, 4))
    plt.plot(steps, losses, label="Train Loss", color="green")
    plt.xlabel("Step")
    plt.ylabel("Loss")
    plt.title(f"Experiment 2 — GPT-2 Top {TOP_N_BLOCKS} Blocks + Embeddings\nTrain Loss Curve")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, "exp2_loss_curve.png"))
    plt.show()
    print("Loss curve saved ✅")

print("\n📁 Outputs saved to:", OUTPUT_PATH)
print("➡️  Next: Run exp3_unfreeze_gpt2_top8.py")